# scicp — Fine-tune MiniLM on Scripture

Fine-tunes `all-MiniLM-L6-v2` on 62k topical-guide (topic → verse) pairs.
With a T4 GPU this takes **~15–20 minutes**.

**Steps:**
1. Run all cells top to bottom
2. Upload `training-pairs.json` when prompted (or mount Google Drive)
3. The final cell downloads `scripture-minilm.zip` — unzip into `resources/models/`
4. Run `python3 scripts/rebake-embeddings.py` on your machine to re-encode all verses

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q sentence-transformers datasets accelerate

In [ ]:
# ── 2. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 3. Upload training-pairs.json ────────────────────────────────────────────
# Option A: upload from your machine
from google.colab import files
uploaded = files.upload()   # select resources/training-pairs.json

# Option B: if you prefer Google Drive, comment out the two lines above and run:
# from google.colab import drive
# drive.mount('/content/drive')
# then set PAIRS_PATH = '/content/drive/MyDrive/training-pairs.json' below

In [ ]:
# ── 4. Load pairs ────────────────────────────────────────────────────────────
import json, random

PAIRS_PATH = 'training-pairs.json'   # change if using Drive

with open(PAIRS_PATH) as f:
    pairs = json.load(f)

random.seed(42)
random.shuffle(pairs)

print(f'Loaded {len(pairs):,} pairs')
print('Sample:', pairs[0])

In [ ]:
# ── 5. Fine-tune ─────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from datasets import Dataset
import time

BASE_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
OUT_DIR     = '/content/scripture-minilm'
BATCH_SIZE  = 128    # T4 can handle 128 comfortably for MiniLM-L6
EPOCHS      = 4
WARMUP_FRAC = 0.1

split = int(len(pairs) * 0.9)
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

train_ds = Dataset.from_dict({
    'anchor':   [p['anchor']   for p in train_pairs],
    'positive': [p['positive'] for p in train_pairs],
})
val_ds = Dataset.from_dict({
    'anchor':   [p['anchor']   for p in val_pairs],
    'positive': [p['positive'] for p in val_pairs],
})

print(f'train={len(train_ds):,}  val={len(val_ds):,}')

model = SentenceTransformer(BASE_MODEL)
loss  = losses.MultipleNegativesRankingLoss(model)

steps_per_epoch = len(train_ds) // BATCH_SIZE
warmup_steps    = int(steps_per_epoch * EPOCHS * WARMUP_FRAC)

args = SentenceTransformerTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=warmup_steps,
    eval_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),   # fp16 on GPU, fp32 on CPU
    bf16=False,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
)

t0 = time.time()
trainer.train()
print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── 6. Save + download ───────────────────────────────────────────────────────
import shutil
from google.colab import files

model.save(OUT_DIR)
print(f'Model saved to {OUT_DIR}')

# Zip and download
zip_path = '/content/scripture-minilm.zip'
shutil.make_archive('/content/scripture-minilm', 'zip', OUT_DIR)
print('Downloading scripture-minilm.zip…')
files.download(zip_path)

## After download

On your local machine:

```bash
# 1. Unzip into resources/models/
mkdir -p resources/models/scripture-minilm
unzip ~/Downloads/scripture-minilm.zip -d resources/models/scripture-minilm

# 2. Re-encode all 41k verses with the fine-tuned model
python3 scripts/rebake-embeddings.py

# 3. Also rebuild cluster labels (centroids are now different)
node scripts/prebake-cluster-labels.js

# 4. Restart the server
npm run dev
```

The server loads embeddings from `verse-embeddings.db` at startup — no code change needed.